# Entrenamiento definitivo v1 · ImageCAS1000 · 100 épocas

Versión diseñada para evitar el cuello de botella observado al entrenar directamente desde Google Drive.

**Configuración científica:**
- Dataset 503 · ImageCAS1000
- `3d_fullres`
- Fold 0
- v1: Dice + CE + 0.2 · soft-clDice
- 10 iteraciones de soft-skeletonization
- 100 épocas
- Deep supervision idéntica a v1
- Resultados y checkpoints guardados en Google Drive

**Cambio de infraestructura:** los datos preprocesados se copian previamente al SSD local de Colab (`/content/nnUNet_preprocessed`). El entrenamiento lee desde el disco local, mientras que los resultados se siguen escribiendo en Drive.

**Split:** no se reutiliza el `splits_final.json` creado cuando el conjunto contenía 999 casos. nnU-Net genera un nuevo 5-fold sobre los 1000 casos y, posteriormente, se guarda una copia del split en Drive para garantizar la reproducibilidad.


## 0. Comprobación de la GPU


In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Activa una GPU antes de continuar.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")


PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.09 GB


## 1. Instalación de dependencias

Si la instalación actualiza paquetes ya cargados, Colab puede solicitar el reinicio de la sesión. En ese caso, se reinicia y se continúa desde el montaje de Drive.


In [2]:
!python -m pip install -q nnunetv2 medpy scikit-image SimpleITK
print("Dependencias instaladas.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 13.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 137.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1b. Reinicio del runtime tras la instalación

`torch` ya se importó en la celda de comprobación de la GPU. Si `pip install` ha actualizado `numpy`, `torch`, `batchgenerators` u otra dependencia de nnU-Net, continuar sin reiniciar expone a un runtime con módulos C cargados en memoria que no coinciden con lo recién instalado (fallo silencioso y difícil de diagnosticar posteriormente).

La ejecución de la siguiente celda reinicia el kernel (aparece un mensaje de sesión terminada). Es el comportamiento previsto: la ejecución continúa desde la celda de montaje de Drive.


In [ ]:
import os
os.kill(os.getpid(), 9)


## 2. Montaje de Google Drive


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


## 3. Definición de rutas y configuración


In [2]:
import os, shutil, json, re, time
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

# Fuente persistente en Drive
DRIVE_PREP_ROOT = BASE / "nnUNet_preprocessed"
DRIVE_DATASET = DRIVE_PREP_ROOT / "Dataset503_ImageCAS1000"

# Copia rápida local para el entrenamiento
LOCAL_PREP_ROOT = Path("/content/nnUNet_preprocessed")
LOCAL_DATASET = LOCAL_PREP_ROOT / "Dataset503_ImageCAS1000"

# Los resultados y checkpoints permanecen en Drive
NNUNET_RESULTS = BASE / "nnUNet_results"
CUSTOM_TRAINERS = BASE / "custom_trainers"
CUSTOM_TRAINERS.mkdir(parents=True, exist_ok=True)

DATASET_ID = 503
DATASET_FOLDER_NAME = "Dataset503_ImageCAS1000"
CONFIGURATION = "3d_fullres"
FOLD = 0

WEIGHT_DICE = 1.0
WEIGHT_CE = 1.0
LAMBDA_CLDICE = 0.2
SKELETON_ITERATIONS = 10
CLASS_NAME = "nnUNetTrainerV1_1000casos_100epochs"

print("Drive dataset:", DRIVE_DATASET)
print("Local dataset:", LOCAL_DATASET)
print("Resultados:", NNUNET_RESULTS)


Drive dataset: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000
Local dataset: /content/nnUNet_preprocessed/Dataset503_ImageCAS1000
Resultados: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results


## 4. Verificación de que Dataset503 en Drive está completo (1000/1000)


In [3]:
drive_cfg = DRIVE_DATASET / "nnUNetPlans_3d_fullres"
drive_gt = DRIVE_DATASET / "gt_segmentations"

imgs = [p for p in drive_cfg.glob("ImageCAS_*.b2nd") if not p.name.endswith("_seg.b2nd")]
segs = list(drive_cfg.glob("ImageCAS_*_seg.b2nd"))
pkls = list(drive_cfg.glob("ImageCAS_*.pkl"))
gts = list(drive_gt.glob("ImageCAS_*.nii.gz"))

print("Imágenes:", len(imgs))
print("Segmentaciones:", len(segs))
print("PKL:", len(pkls))
print("GT:", len(gts))

for required in ["nnUNetPlans.json", "dataset.json", "dataset_fingerprint.json"]:
    p = DRIVE_DATASET / required
    print(required, "->", p.exists())

assert len(imgs) == 1000, f"Hay {len(imgs)} imágenes, no 1000."
assert len(segs) == 1000, f"Hay {len(segs)} segmentaciones, no 1000."
assert len(pkls) == 1000, f"Hay {len(pkls)} PKL, no 1000."
assert len(gts) == 1000, f"Hay {len(gts)} GT, no 1000."
assert all((DRIVE_DATASET/x).exists() for x in ["nnUNetPlans.json","dataset.json","dataset_fingerprint.json"])

print("\n✅ Dataset503 en Drive: 1000/1000 y archivos de planificación presentes.")


Imágenes: 1000
Segmentaciones: 1000
PKL: 1000
GT: 1000
nnUNetPlans.json -> True
dataset.json -> True
dataset_fingerprint.json -> True

✅ Dataset503 en Drive: 1000/1000 y archivos de planificación presentes.


## 5. Comprobación del espacio local

Se requiere un espacio libre considerable, ya que `3d_fullres` se copia desde Drive al SSD local. La celda no recorre Drive.


In [4]:
usage = shutil.disk_usage("/content")
free_gb = usage.free / 1024**3
print(f"Espacio libre local: {free_gb:.1f} GB")

if free_gb < 120:
    raise RuntimeError(
        f"Solo hay {free_gb:.1f} GB libres. No copies todavía el dataset. "
        "Necesitamos margen suficiente para preprocesados y temporales."
    )
print("✅ Espacio local suficiente para intentar la copia.")


Espacio libre local: 187.8 GB
✅ Espacio local suficiente para intentar la copia.


## 6. Copia desde Drive al SSD local de los archivos necesarios

No se copian resultados antiguos ni otras configuraciones. Se copian:
- `nnUNetPlans_3d_fullres`
- `gt_segmentations`
- `nnUNetPlans.json`
- `dataset.json`
- `dataset_fingerprint.json`

Se excluye deliberadamente `splits_final.json`, ya que el existente en Drive puede corresponder al intento anterior con 999 casos.

**Nota:** el `splits_final.json` antiguo (999 casos) permanece sin modificar en `MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000/splits_final.json`. No interfiere mientras el entrenamiento se realice siempre a través de este notebook (que nunca lo copia), pero se volvería a utilizar si `nnUNet_preprocessed` apuntara directamente a Drive sin pasar por la copia local. Para evitar confusiones, se recomienda renombrarlo a `splits_final_OBSOLETO_999casos.json`.


In [5]:
if LOCAL_DATASET.exists():
    print("Ya existe una copia local. Se elimina para evitar mezclar una copia incompleta/antigua.")
    shutil.rmtree(LOCAL_DATASET)

LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

# Archivos raíz
for name in ["nnUNetPlans.json", "dataset.json", "dataset_fingerprint.json"]:
    src = DRIVE_DATASET / name
    dst = LOCAL_DATASET / name
    print("Copiando:", name)
    shutil.copy2(src, dst)

# Carpetas grandes
for folder_name in ["nnUNetPlans_3d_fullres", "gt_segmentations"]:
    src_dir = DRIVE_DATASET / folder_name
    dst_dir = LOCAL_DATASET / folder_name
    print(f"\nCopiando {folder_name} ...")
    t0 = time.time()
    shutil.copytree(src_dir, dst_dir)
    print(f"✓ {folder_name} copiado en {(time.time()-t0)/60:.1f} min")

print("\n✅ Copia local terminada.")


Copiando: nnUNetPlans.json
Copiando: dataset.json
Copiando: dataset_fingerprint.json

Copiando nnUNetPlans_3d_fullres ...
✓ nnUNetPlans_3d_fullres copiado en 269.9 min

Copiando gt_segmentations ...
✓ gt_segmentations copiado en 1.5 min

✅ Copia local terminada.


## 7. Verificación de la copia local y de que no incluye el split de 999 casos


In [6]:
local_cfg = LOCAL_DATASET / "nnUNetPlans_3d_fullres"
local_gt = LOCAL_DATASET / "gt_segmentations"

imgs_local = [p for p in local_cfg.glob("ImageCAS_*.b2nd") if not p.name.endswith("_seg.b2nd")]
segs_local = list(local_cfg.glob("ImageCAS_*_seg.b2nd"))
pkls_local = list(local_cfg.glob("ImageCAS_*.pkl"))
gts_local = list(local_gt.glob("ImageCAS_*.nii.gz"))

print("Imágenes locales:", len(imgs_local))
print("Segmentaciones locales:", len(segs_local))
print("PKL locales:", len(pkls_local))
print("GT locales:", len(gts_local))
print("splits_final.json local existe:", (LOCAL_DATASET/"splits_final.json").exists())

assert len(imgs_local) == 1000
assert len(segs_local) == 1000
assert len(pkls_local) == 1000
assert len(gts_local) == 1000
assert not (LOCAL_DATASET/"splits_final.json").exists()

print("\n✅ Copia local íntegra: 1000/1000.")
print("✅ Sin split antiguo: nnU-Net generará un nuevo 5-fold sobre los 1000 casos.")


Imágenes locales: 1000
Segmentaciones locales: 1000
PKL locales: 1000
GT locales: 1000
splits_final.json local existe: False

✅ Copia local íntegra: 1000/1000.
✅ Sin split antiguo: nnU-Net generará un nuevo 5-fold sobre los 1000 casos.


## 8. Configuración de nnU-Net: datos en el SSD local y resultados en Drive


In [7]:
os.environ["nnUNet_preprocessed"] = str(LOCAL_PREP_ROOT)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)

# raw no es necesario para entrenar; se deja una ruta válida si existe
NNUNET_RAW = BASE / "nnUNet_raw"
os.environ["nnUNet_raw"] = str(NNUNET_RAW)

# GT_FOLDER apunta a Drive (persistente) y no a la copia local efímera:
# si la sesión se desconecta tras entrenar, /content deja de existir pero Drive se conserva,
# lo que permite evaluar posteriormente
GT_FOLDER = DRIVE_DATASET / "gt_segmentations"

print("nnUNet_preprocessed =", os.environ["nnUNet_preprocessed"])
print("nnUNet_results      =", os.environ["nnUNet_results"])
print("nnUNet_extTrainer   =", os.environ["nnUNet_extTrainer"])
print("\n✅ Entrenamiento leerá preprocesados del SSD local.")


nnUNet_preprocessed = /content/nnUNet_preprocessed
nnUNet_results      = /content/drive/MyDrive/TFM_nnUNet/nnUNet_results
nnUNet_extTrainer   = /content/drive/MyDrive/TFM_nnUNet/custom_trainers

✅ Entrenamiento leerá preprocesados del SSD local.


## 9. Funciones de métricas


In [8]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)


def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing


def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)


def hard_cldice(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    return (2 * tprec * tsens) / (tprec + tsens + 1e-8)


def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n


def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n


def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)


def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)
    return {
        "dice": dice_score(pred, gt),
        "cldice": hard_cldice(pred, gt),
        "cobertura_principal": cobertura,
        "n_componentes": n_componentes,
        "hd95_crudo_mm": hd95_seguro(pred, gt, spacing),
    }


def evaluar_trainer(trainer_name):
    carpeta_val = (
        NNUNET_RESULTS / DATASET_FOLDER_NAME
        / f"{trainer_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}" / "validation"
    )
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    filas = []
    for pred_path in preds:
        case_id = pred_path.name.replace(".nii.gz", "")
        gt_path = GT_FOLDER / pred_path.name
        if not gt_path.exists():
            continue
        m = calcular_metricas_caso(pred_path, gt_path)
        m["case_id"] = case_id
        filas.append(m)

    df = pd.DataFrame(filas)
    resumen = {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }
    return df, resumen

print("Funciones de metricas cargadas.")


Funciones de metricas cargadas.


## 10. Trainer v1 idéntico al del experimento de 200 casos


In [9]:
TRAINER_TEMPLATE = r'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_{epochs}epochs
)


def soft_erode_3d(x):
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)
    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


class SoftClDiceLoss(nn.Module):
    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()
        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)
        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):
    """Identico a v1 (notebook 12): combina Dice+CE+clDice en un unico modulo
    por escala; DeepSupervisionWrapper envuelve este modulo COMPLETO, asi que
    clDice se recalcula de forma independiente en cada escala de deep supervision."""

    def __init__(self, regional_loss, lambda_cldice=0.2, skeleton_iterations=10):
        super().__init__()
        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice
        self.soft_cldice = SoftClDiceLoss(iterations=skeleton_iterations, smooth=1e-5)

    def forward(self, logits, target):
        loss_regional = self.regional_loss(logits, target)

        probabilities = torch.softmax(logits.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]

        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)
        foreground_target = (target == 1).float()

        topology_loss = self.soft_cldice(foreground_probability, foreground_target)

        return loss_regional + self.lambda_cldice * topology_loss


class {class_name}(nnUNetTrainer_{epochs}epochs):

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.weight_dice = {weight_dice}
        self.weight_ce = {weight_ce}
        self.lambda_cldice = {lambda_cldice}
        self.skeleton_iterations = {skeleton_iterations}

    def on_train_start(self):
        super().on_train_start()
        self.print_to_log_file(
            f"[v1 escalado a 1000 casos] weight_dice={{self.weight_dice}} "
            f"weight_ce={{self.weight_ce}} lambda_cldice={{self.lambda_cldice}} "
            f"skeleton_iterations={{self.skeleton_iterations}}"
        )

    def _build_loss(self):
        regional_loss = DC_and_CE_loss(
            soft_dice_kwargs={{
                "batch_dice": self.configuration_manager.batch_dice,
                "smooth": 1e-5,
                "do_bg": False,
                "ddp": self.is_ddp
            }},
            ce_kwargs={{}},
            weight_ce=self.weight_ce,
            weight_dice=self.weight_dice,
            ignore_label=self.label_manager.ignore_label,
            dice_class=MemoryEfficientSoftDiceLoss
        )

        loss = DiceCEPlusSoftClDice(
            regional_loss=regional_loss,
            lambda_cldice=self.lambda_cldice,
            skeleton_iterations=self.skeleton_iterations,
        )

        if self.enable_deep_supervision:
            deep_supervision_scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** i) for i in range(len(deep_supervision_scales))])
            weights[-1] = 0
            weights = weights / weights.sum()
            loss = DeepSupervisionWrapper(loss, weights)

        return loss
'''

codigo = TRAINER_TEMPLATE.format(
    class_name=CLASS_NAME,
    epochs=100,
    weight_dice=WEIGHT_DICE,
    weight_ce=WEIGHT_CE,
    lambda_cldice=LAMBDA_CLDICE,
    skeleton_iterations=SKELETON_ITERATIONS,
)
ruta_trainer = CUSTOM_TRAINERS / f"{CLASS_NAME}.py"
ruta_trainer.write_text(codigo)
print("Trainer escrito en:", ruta_trainer)


Trainer escrito en: /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerV1_1000casos_100epochs.py


## 11. Comprobación final previa al entrenamiento

La salida debe mostrar:
- 1000 imágenes locales
- 1000 segmentaciones locales
- `splits_final.json` inexistente
- Trainer existente

Al iniciar el entrenamiento, nnU-Net crea un nuevo **5-fold de 1000 casos**. Para el fold 0 se esperan 800 casos de entrenamiento y 200 de validación.


In [10]:
print("=== CHECK FINAL ===")
print("Dataset local:", LOCAL_DATASET)
print("Imágenes:", len(imgs_local))
print("Segmentaciones:", len(segs_local))
print("Trainer:", CUSTOM_TRAINERS / f"{CLASS_NAME}.py")
print("Trainer existe:", (CUSTOM_TRAINERS / f"{CLASS_NAME}.py").exists())
print("Split local antes de entrenar:", (LOCAL_DATASET/"splits_final.json").exists())

assert len(imgs_local) == 1000 and len(segs_local) == 1000
assert (CUSTOM_TRAINERS / f"{CLASS_NAME}.py").exists()
assert not (LOCAL_DATASET/"splits_final.json").exists()

print("\n✅ LISTO PARA ENTRENAR.")


=== CHECK FINAL ===
Dataset local: /content/nnUNet_preprocessed/Dataset503_ImageCAS1000
Imágenes: 1000
Segmentaciones: 1000
Trainer: /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerV1_1000casos_100epochs.py
Trainer existe: True
Split local antes de entrenar: False

✅ LISTO PARA ENTRENAR.


## 12. Entrenamiento (100 épocas)

Al iniciarse, la salida debe indicar un nuevo split de 1000 casos y alcanzar `Epoch 0`. Las celdas posteriores se ejecutan una vez finalizado el entrenamiento.


In [11]:
!nnUNetv2_train \
    503 \
    3d_fullres \
    0 \
    -tr nnUNetTrainerV1_1000casos_100epochs \
    --npz



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Trainer 'nnUNetTrainerV1_1000casos_100epochs' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Searching for class nnUNetTrainerV1_1000casos_100epochs in folder /content/drive/MyDrive/TFM_nnUNet/custom_trainers with current module None
  Inspecting module: nnUNetTrainerLambdaV1Fiel_100epochs
  Inspecting module: nnUNetTrainerOptunaBest_100epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial0_50epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial1_50epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial2_50epochs
  Inspecting module: nnUN

## 12b. Desconexión de Colab durante el entrenamiento

Incluida la validación, el entrenamiento completo puede superar las 3 h, cerca del límite de sesión de Colab. En caso de desconexión:

1. Se vuelve a ejecutar desde el montaje de Drive (sección 2) hasta la copia local y su verificación (secciones 3-8). La copia es determinista: reproduce el mismo `Dataset503_ImageCAS1000` local, incluido el `splits_final.json` que nnU-Net creó en la copia local durante el primer intento. Este archivo **no debe eliminarse**: es el split de 1000 casos generado al arrancar y `--c` lo necesita para continuar con la misma partición de entrenamiento y validación.
2. En lugar de la celda de entrenamiento original, se ejecuta la celda siguiente (con `--c`).

Si la copia local se ha perdido por completo (nuevo runtime) y es necesario regenerarla, debe tenerse en cuenta que el paso 6 tal como está escrito **elimina** cualquier `LOCAL_DATASET` previo. Para continuar el entrenamiento, no se repite la celda de borrado y copia una vez existe el split local, o bien se realiza una copia de seguridad del `splits_final.json` local antes de volver a copiar.


In [12]:
# Solo si el entrenamiento anterior se interrumpió y se continúa
# desde el último checkpoint guardado en Drive, con el mismo split:
#!nnUNetv2_train \
#    503 \
#    3d_fullres \
#    0 \
#    -tr nnUNetTrainerV1_1000casos_100epochs \
#    --npz \
#    --c



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Trainer 'nnUNetTrainerV1_1000casos_100epochs' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Searching for class nnUNetTrainerV1_1000casos_100epochs in folder /content/drive/MyDrive/TFM_nnUNet/custom_trainers with current module None
  Inspecting module: nnUNetTrainerLambdaV1Fiel_100epochs
  Inspecting module: nnUNetTrainerOptunaBest_100epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial0_50epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial1_50epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial2_50epochs
  Inspecting module: nnUN

## 13. Guardado del split exacto utilizado (reproducibilidad)


In [13]:
split_local = LOCAL_DATASET / "splits_final.json"
assert split_local.exists(), "No encuentro el splits_final.json generado por nnU-Net."

with open(split_local, "r") as f:
    splits = json.load(f)

print("Número de folds:", len(splits))
for i, sp in enumerate(splits):
    print(f"Fold {i}: train={len(sp['train'])}, val={len(sp['val'])}")

split_backup = BASE / "splits_final_Dataset503_1000_auto5fold.json"
shutil.copy2(split_local, split_backup)
print("\n✅ Split utilizado guardado en:", split_backup)


Número de folds: 5
Fold 0: train=800, val=200
Fold 1: train=800, val=200
Fold 2: train=800, val=200
Fold 3: train=800, val=200
Fold 4: train=800, val=200

✅ Split utilizado guardado en: /content/drive/MyDrive/TFM_nnUNet/splits_final_Dataset503_1000_auto5fold.json


## 14. Evaluación y guardado de métricas caso a caso


In [14]:
df_metricas, resumen = evaluar_trainer(CLASS_NAME)

print("=== Resultados: v1 (lambda=0.2), Dataset503, fold 0, 100 épocas ===")
print(f"Dice:          {resumen['dice_mean']:.4f}")
print(f"clDice:        {resumen['cldice_mean']:.4f}")
print(f"n_componentes: {resumen['n_componentes_mean']:.2f}")
print(f"HD95 crudo:    {resumen['hd95_crudo_mean']:.2f} mm")
print(f"n_casos:       {resumen['n_casos']}")

csv_out = BASE / "v1_1000casos_100epochs_metricas_por_caso.csv"
df_metricas.to_csv(csv_out, index=False)
print("\nGuardado:", csv_out)


=== Resultados: v1 (lambda=0.2), Dataset503, fold 0, 100 épocas ===
Dice:          0.7475
clDice:        0.8113
n_componentes: 16.98
HD95 crudo:    24.72 mm
n_casos:       200

Guardado: /content/drive/MyDrive/TFM_nnUNet/v1_1000casos_100epochs_metricas_por_caso.csv


## 15. Comparación descriptiva con el experimento de 200 casos

**Nota:** el HD95 de esta función utiliza `medpy_hd95`. Debe mantenerse la misma implementación en las comparaciones, ya que existe una discrepancia conocida con una implementación anterior de HD95 del proyecto.


In [15]:
comparativa = pd.DataFrame([
    {
        "modelo": "v1 λ=.2 (200 casos)",
        "dice": 0.7549,
        "cldice": 0.8221,
        "n_componentes": 14.725,
        "hd95_mm_medpy": 21.0636,
    },
    {
        "modelo": "v1 λ=.2 (Dataset503 1000)",
        "dice": resumen["dice_mean"],
        "cldice": resumen["cldice_mean"],
        "n_componentes": resumen["n_componentes_mean"],
        "hd95_mm_medpy": resumen["hd95_crudo_mean"],
    },
])
print(comparativa.to_string(index=False))


                   modelo     dice   cldice  n_componentes  hd95_mm_medpy
      v1 λ=.2 (200 casos) 0.754900 0.822100         14.725      21.063600
v1 λ=.2 (Dataset503 1000) 0.747547 0.811284         16.985      24.719474


## Archivos persistentes

- Checkpoints y resultados → `MyDrive/TFM_nnUNet/nnUNet_results`
- Métricas → `v1_1000casos_100epochs_metricas_por_caso.csv`
- Split exacto → `splits_final_Dataset503_1000_auto5fold.json`

La copia de `/content/nnUNet_preprocessed` es temporal y desaparece al cerrar el runtime; los datos originales de Drive no se modifican.
